# Problema 2

Puede que se vea largo, denso, la idea es no asustarse. Es bueno que vean problemas largos, porque en los laboratorios como pueden ver tampoco son ejercicios pequeños.
Aquí hay varias decisiones de diseño que se pueden tomar, y no hay una única forma correcta de hacerlo. Lo importante es que el código sea claro, y que cumpla con los requerimientos.

## 1. Modelo POO

In [16]:
from pathlib import Path
import json
import random


class Persona:
    def __init__(self, id_persona, nombre, email):
        self.id_persona = id_persona
        self.nombre = nombre
        self.email = email

    def to_dict(self):
        return {
            "id_persona": self.id_persona,
            "nombre": self.nombre,
            "email": self.email,
        }


class Estudiante(Persona):
    def __init__(
        self,
        id_persona,
        nombre,
        email,
        carrera,
        anio_ingreso,
        max_creditos=30,
        cursos_aprobados=None,
    ):
        super().__init__(id_persona, nombre, email)
        self.carrera = carrera
        self.anio_ingreso = anio_ingreso
        self.max_creditos = max_creditos
        self.cursos_aprobados = cursos_aprobados if cursos_aprobados is not None else set() # nuevamente el set es algo que es bueno estudiar.
        self.inscripciones_activas = set()

    def to_dict(self):
        data = super().to_dict() # estudien bien el tema del super.
        data.update({
            "carrera": self.carrera,
            "anio_ingreso": self.anio_ingreso,
            "max_creditos": self.max_creditos,
            "cursos_aprobados": sorted(self.cursos_aprobados),
        })
        return data


class Profesor(Persona):
    def __init__(
        self,
        id_persona,
        nombre,
        email,
        departamento,
        max_creditos_docencia=24,
    ):
        super().__init__(id_persona, nombre, email)
        self.departamento = departamento
        self.max_creditos_docencia = max_creditos_docencia

    def to_dict(self):
        data = super().to_dict()
        data.update({
            "departamento": self.departamento,
            "max_creditos_docencia": self.max_creditos_docencia,
        })
        return data


class Curso:
    def __init__(self, codigo, nombre, creditos, prerrequisitos=None):
        self.codigo = codigo
        self.nombre = nombre
        self.creditos = creditos
        self.prerrequisitos = prerrequisitos if prerrequisitos is not None else set()

    def agregar_prerrequisito(self, codigo_curso):
        self.prerrequisitos.add(codigo_curso)

    def to_dict(self):
        return {
            "codigo": self.codigo,
            "nombre": self.nombre,
            "creditos": self.creditos,
            "prerrequisitos": sorted(self.prerrequisitos),
        }


class Horario:
    dias_validos = {"lunes", "martes", "miercoles", "jueves", "viernes"}

    def __init__(self, dia, hora_inicio, hora_fin):
        if dia not in self.dias_validos:
            raise ValueError(f"Dia invalido: {dia}")
        if hora_inicio >= hora_fin:
            raise ValueError("La hora de inicio debe ser menor que la hora de termino")
        self.dia = dia
        self.hora_inicio = hora_inicio
        self.hora_fin = hora_fin

    def tiene_conflicto(self, otro):
        mismo_dia = self.dia == otro.dia
        se_cruzan = self.hora_inicio < otro.hora_fin and otro.hora_inicio < self.hora_fin
        return mismo_dia and se_cruzan

    def to_dict(self):
        return {
            "dia": self.dia,
            "hora_inicio": self.hora_inicio,
            "hora_fin": self.hora_fin,
        }

    def __str__(self):
        return f"{self.dia} {self.hora_inicio}:00-{self.hora_fin}:00"


class Seccion:
    def __init__(
        self,
        id_seccion,
        codigo_curso,
        capacidad,
        horarios,
        profesor_id,
    ):
        self.id_seccion = id_seccion
        self.codigo_curso = codigo_curso
        self.capacidad = capacidad
        self.horarios = horarios
        self.profesor_id = profesor_id
        self.estudiantes = set()
        self.movimientos = 0

    def cupos_disponibles(self):
        return self.capacidad - len(self.estudiantes)

    def esta_llena(self):
        return self.cupos_disponibles() <= 0

    def tiene_conflicto_horario(self, otra):
        for horario in self.horarios:
            for otro_horario in otra.horarios:
                if horario.tiene_conflicto(otro_horario):
                    return True
        return False

    def to_dict(self):
        return {
            "id_seccion": self.id_seccion,
            "codigo_curso": self.codigo_curso,
            "capacidad": self.capacidad,
            "horarios": [h.to_dict() for h in self.horarios],
            "profesor_id": self.profesor_id,
        }


class Inscripcion:
    def __init__(self, id_inscripcion, estudiante_id, seccion_id, estado = "activa"):
        self.id_inscripcion = id_inscripcion
        self.estudiante_id = estudiante_id
        self.seccion_id = seccion_id
        self.estado = estado

    def cancelar(self):
        self.estado = "cancelada"

    def to_dict(self):
        return {
            "id_inscripcion": self.id_inscripcion,
            "estudiante_id": self.estudiante_id,
            "seccion_id": self.seccion_id,
            "estado": self.estado,
        }


class SolicitudInscripcion:
    def __init__(
        self,
        id_solicitud,
        estudiante_id,
        seccion_id,
        tipo = "inscribir",
        estado = "pendiente",
        motivo = "",
    ):
        self.id_solicitud = id_solicitud
        self.estudiante_id = estudiante_id
        self.seccion_id = seccion_id
        self.tipo = tipo
        self.estado = estado
        self.motivo = motivo

    def to_dict(self):
        return {
            "id_solicitud": self.id_solicitud,
            "estudiante_id": self.estudiante_id,
            "seccion_id": self.seccion_id,
            "tipo": self.tipo,
        }


def estudiante_desde_dict(data):
    return Estudiante(
        id_persona=data["id_persona"],
        nombre=data["nombre"],
        email=data["email"],
        carrera=data["carrera"],
        anio_ingreso=data["anio_ingreso"],
        max_creditos=data.get("max_creditos", 30),
        cursos_aprobados=set(data.get("cursos_aprobados", [])),
    )


def profesor_desde_dict(data):
    return Profesor(
        id_persona=data["id_persona"],
        nombre=data["nombre"],
        email=data["email"],
        departamento=data["departamento"],
        max_creditos_docencia=data.get("max_creditos_docencia", 24),
    )


def curso_desde_dict(data):
    return Curso(
        codigo=data["codigo"],
        nombre=data["nombre"],
        creditos=data["creditos"],
        prerrequisitos=set(data.get("prerrequisitos", [])),
    )


def horario_desde_dict(data):
    return Horario(data["dia"], data["hora_inicio"], data["hora_fin"])


def seccion_desde_dict(data):
    return Seccion(
        id_seccion=data["id_seccion"],
        codigo_curso=data["codigo_curso"],
        capacidad=data["capacidad"],
        horarios=[horario_desde_dict(h) for h in data.get("horarios", [])],
        profesor_id=data.get("profesor_id"),
    )


def solicitud_desde_dict(data):
    return SolicitudInscripcion(
        id_solicitud=data["id_solicitud"],
        estudiante_id=data["estudiante_id"],
        seccion_id=data["seccion_id"],
        tipo=data.get("tipo", "inscribir"),
    )

## 2. Sistema academico y validación

Este sistema centraliza los diccionarios de objetos y valida cada proceso, que pueden ser las secciones existentes, prerrequisitos, choques de horario, cupos, carga maxima del estudiante y carga docente del profesor.

In [17]:
class SistemaAcademico:
    def __init__(self, semestre):
        self.semestre = semestre
        self.estudiantes = {} 
        self.profesores = {}
        self.cursos = {}
        self.secciones = {}
        self.inscripciones = {}
        self.solicitudes = {}
        self.bitacora= []
        self._contador_inscripciones = 1

    def agregar_estudiante(self, estudiante):
        self.estudiantes[estudiante.id_persona] = estudiante

    def agregar_profesor(self, profesor):
        self.profesores[profesor.id_persona] = profesor

    def agregar_curso(self, curso):
        self.cursos[curso.codigo] = curso

    def agregar_seccion(self, seccion):
        if seccion.codigo_curso not in self.cursos:
            raise ValueError(f"No existe el curso {seccion.codigo_curso}")
        self.secciones[seccion.id_seccion] = seccion

    def _registrar(self, evento, estado, detalle, **extra):
        registro = {
            "paso": len(self.bitacora) + 1,
            "evento": evento,
            "estado": estado,
            "detalle": detalle,
        }
        registro.update(extra)
        self.bitacora.append(registro)

    def _horarios_compatibles(self, seccion, secciones_actuales): # el guion es un metodo interno. No es imprescindible que lo usen, pero es una buena practica. Sirve saberlo.
        for otra in secciones_actuales:
            if seccion.tiene_conflicto_horario(otra):
                return False
        return True

    def _inscripciones_activas_de(self, estudiante_id):
        activas = []
        for inscripcion in self.inscripciones.values():
            if inscripcion.estudiante_id == estudiante_id and inscripcion.estado == "activa":
                activas.append(inscripcion)
        return activas

    def carga_creditos_estudiante(self, estudiante_id):
        total = 0
        for inscripcion in self._inscripciones_activas_de(estudiante_id):
            seccion = self.secciones[inscripcion.seccion_id]
            curso = self.cursos[seccion.codigo_curso]
            total += curso.creditos
        return total

    def carga_profesor(self, profesor_id):
        total = 0
        for seccion in self.secciones.values():
            if seccion.profesor_id == profesor_id:
                curso = self.cursos[seccion.codigo_curso]
                total += curso.creditos
        return total

    def asignar_profesor(self, profesor_id, seccion_id):
        if profesor_id not in self.profesores:
            detalle = f"Profesor inexistente: {profesor_id}"
            self._registrar("asignar_profesor", "rechazado", detalle, profesor=profesor_id, seccion=seccion_id)
            return False, detalle
        if seccion_id not in self.secciones:
            detalle = f"Seccion inexistente: {seccion_id}"
            self._registrar("asignar_profesor", "rechazado", detalle, profesor=profesor_id, seccion=seccion_id)
            return False, detalle

        profesor = self.profesores[profesor_id]
        seccion = self.secciones[seccion_id]
        curso = self.cursos[seccion.codigo_curso]
        secciones_actuales = []
        for otra in self.secciones.values():
            if otra.profesor_id == profesor_id:
                secciones_actuales.append(otra)

        if seccion.profesor_id is not None:
            detalle = f"La seccion {seccion_id} ya tiene profesor asignado"
            self._registrar("asignar_profesor", "rechazado", detalle, profesor=profesor_id, seccion=seccion_id, curso=curso.codigo)
            return False, detalle
        if not self._horarios_compatibles(seccion, secciones_actuales):
            detalle = f"Choque de horario para profesor {profesor_id}"
            self._registrar("asignar_profesor", "rechazado", detalle, profesor=profesor_id, seccion=seccion_id, curso=curso.codigo)
            return False, detalle
        if self.carga_profesor(profesor_id) + curso.creditos > profesor.max_creditos_docencia:
            detalle = f"Carga docente maxima excedida para {profesor_id}"
            self._registrar("asignar_profesor", "rechazado", detalle, profesor=profesor_id, seccion=seccion_id, curso=curso.codigo)
            return False, detalle

        seccion.profesor_id = profesor_id
        detalle = f"Profesor {profesor_id} asignado a {seccion_id}"
        self._registrar("asignar_profesor", "aceptado", detalle, profesor=profesor_id, seccion=seccion_id, curso=curso.codigo)
        return True, detalle

    def procesar_solicitud(self, solicitud):
        self.solicitudes[solicitud.id_solicitud] = solicitud
        if solicitud.tipo not in {"inscribir", "cancelar"}:
            return self._rechazar_solicitud(solicitud, "Tipo de solicitud invalido")
        if solicitud.estudiante_id not in self.estudiantes:
            return self._rechazar_solicitud(solicitud, "Estudiante inexistente")
        if solicitud.seccion_id not in self.secciones:
            return self._rechazar_solicitud(solicitud, "Seccion inexistente")
        if solicitud.tipo == "cancelar":
            return self._cancelar_inscripcion(solicitud)
        return self._inscribir_estudiante(solicitud)

    def _rechazar_solicitud(self, solicitud, motivo):
        solicitud.estado = "rechazada"
        solicitud.motivo = motivo
        extra = {
            "solicitud": solicitud.id_solicitud,
            "tipo": solicitud.tipo,
            "estudiante": solicitud.estudiante_id,
            "seccion": solicitud.seccion_id,
        }
        if solicitud.seccion_id in self.secciones:
            seccion = self.secciones[solicitud.seccion_id]
            seccion.movimientos += 1
            extra["curso"] = seccion.codigo_curso
        self._registrar("solicitud", "rechazado", motivo, **extra)
        return False, motivo

    def _cancelar_inscripcion(self, solicitud):
        activa = None
        for inscripcion in self._inscripciones_activas_de(solicitud.estudiante_id):
            if inscripcion.seccion_id == solicitud.seccion_id:
                activa = inscripcion
                break
        if activa is None:
            return self._rechazar_solicitud(solicitud, "No existe inscripcion activa para cancelar")

        estudiante = self.estudiantes[solicitud.estudiante_id]
        seccion = self.secciones[solicitud.seccion_id]
        activa.cancelar()
        estudiante.inscripciones_activas.discard(activa.id_inscripcion)
        seccion.estudiantes.discard(estudiante.id_persona)
        seccion.movimientos += 1
        solicitud.estado = "aceptada"
        solicitud.motivo = "Cancelacion procesada"
        detalle = f"{estudiante.id_persona} cancela {seccion.id_seccion}"
        self._registrar(
            "solicitud",
            "aceptado",
            detalle,
            solicitud=solicitud.id_solicitud,
            tipo="cancelar",
            estudiante=estudiante.id_persona,
            seccion=seccion.id_seccion,
            curso=seccion.codigo_curso,
        )
        return True, detalle

    def _inscribir_estudiante(self, solicitud):
        estudiante = self.estudiantes[solicitud.estudiante_id]
        seccion = self.secciones[solicitud.seccion_id]
        curso = self.cursos[seccion.codigo_curso]
        activas = self._inscripciones_activas_de(estudiante.id_persona)

        cursos_activos = set()
        for inscripcion in activas:
            seccion_activa = self.secciones[inscripcion.seccion_id]
            cursos_activos.add(seccion_activa.codigo_curso)

        if curso.codigo in cursos_activos:
            return self._rechazar_solicitud(solicitud, "El estudiante ya tiene una seccion activa de este curso")

        faltantes = curso.prerrequisitos - estudiante.cursos_aprobados
        if faltantes:
            return self._rechazar_solicitud(solicitud, f"Faltan prerrequisitos: {sorted(faltantes)}")

        if self.carga_creditos_estudiante(estudiante.id_persona) + curso.creditos > estudiante.max_creditos:
            return self._rechazar_solicitud(solicitud, "Carga academica maxima excedida")

        secciones_activas = []
        for inscripcion in activas:
            secciones_activas.append(self.secciones[inscripcion.seccion_id])
        if not self._horarios_compatibles(seccion, secciones_activas):
            return self._rechazar_solicitud(solicitud, "Choque de horario con otra inscripcion")

        if seccion.esta_llena():
            return self._rechazar_solicitud(solicitud, "No quedan cupos disponibles")

        id_inscripcion = f"I{self._contador_inscripciones:04d}"
        self._contador_inscripciones += 1
        inscripcion = Inscripcion(id_inscripcion, estudiante.id_persona, seccion.id_seccion)
        self.inscripciones[id_inscripcion] = inscripcion
        estudiante.inscripciones_activas.add(id_inscripcion)
        seccion.estudiantes.add(estudiante.id_persona)
        seccion.movimientos += 1
        solicitud.estado = "aceptada"
        solicitud.motivo = "Inscripcion procesada"
        detalle = f"{estudiante.id_persona} inscrito en {seccion.id_seccion}"
        self._registrar(
            "solicitud",
            "aceptado",
            detalle,
            solicitud=solicitud.id_solicitud,
            tipo="inscribir",
            estudiante=estudiante.id_persona,
            seccion=seccion.id_seccion,
            curso=curso.codigo,
            inscripcion=id_inscripcion,
        )
        return True, detalle

    def cargar_datos_iniciales(self, datos):
        for curso_data in datos.get("cursos", []):
            self.agregar_curso(curso_desde_dict(curso_data))
        for estudiante_data in datos.get("estudiantes", []):
            self.agregar_estudiante(estudiante_desde_dict(estudiante_data))
        for profesor_data in datos.get("profesores", []):
            self.agregar_profesor(profesor_desde_dict(profesor_data))
        for seccion_data in datos.get("secciones", []):
            self.agregar_seccion(seccion_desde_dict(seccion_data))

    def to_dict(self):
        estudiantes = []
        for estudiante_id in sorted(self.estudiantes):
            estudiantes.append(self.estudiantes[estudiante_id].to_dict())

        profesores = []
        for profesor_id in sorted(self.profesores):
            profesores.append(self.profesores[profesor_id].to_dict())

        cursos = []
        for codigo in sorted(self.cursos):
            cursos.append(self.cursos[codigo].to_dict())

        secciones = []
        for seccion_id in sorted(self.secciones):
            secciones.append(self.secciones[seccion_id].to_dict())

        inscripciones = []
        for inscripcion_id in sorted(self.inscripciones):
            inscripciones.append(self.inscripciones[inscripcion_id].to_dict())

        return {
            "semestre": self.semestre,
            "estudiantes": estudiantes,
            "profesores": profesores,
            "cursos": cursos,
            "secciones": secciones,
            "inscripciones": inscripciones,
            "bitacora": self.bitacora,
        }

    def resumen(self):
        inscripciones_activas = 0
        for inscripcion in self.inscripciones.values():
            if inscripcion.estado == "activa":
                inscripciones_activas += 1
        return {
            "semestre": self.semestre,
            "estudiantes": len(self.estudiantes),
            "profesores": len(self.profesores),
            "cursos": len(self.cursos),
            "secciones": len(self.secciones),
            "inscripciones_activas": inscripciones_activas,
            "movimientos": len(self.bitacora),
        }


def sistema_desde_datos(datos):
    sistema = SistemaAcademico(datos["semestre"])
    sistema.cargar_datos_iniciales(datos)
    return sistema

## 3. Generador JSON

Datos iniciales y eventos planificados. Para evitar inconsistencias, las solicitudes aleatorias se aceptan primero en un sistema temporal y solo se guardan si son validas.

In [18]:
cursos_base = [
    {"codigo": "MAT1610", "nombre": "Calculo I", "creditos": 6, "prerrequisitos": []},
    {"codigo": "ICE1514", "nombre": "Dinámica", "creditos": 6, "prerrequisitos": []},
    {"codigo": "IIC1103", "nombre": "Introduccion a la Programacion", "creditos": 6, "prerrequisitos": []},
    {"codigo": "IIC2233", "nombre": "Programacion Avanzada", "creditos": 6, "prerrequisitos": ["IIC1103"]},
    {"codigo": "IIC2133", "nombre": "Estructuras de Datos", "creditos": 6, "prerrequisitos": ["IIC1103"]},
    {"codigo": "EYP1103", "nombre": "Probabilidad y Estadistica", "creditos": 5, "prerrequisitos": ["MAT1610"]},
]

nombres_estudiantes = [
    "Ana Torres", "Benjamin Soto", "Camila Rojas", "Diego Silva", "Elena Fuentes", "Felipe Vidal",
    "Gabriela Munoz", "Hector Castro", "Isidora Vega", "Javier Morales", "Karla Paredes", "Lucas Diaz",
    "Martina Reyes", "Nicolas Herrera", "Olivia Campos", "Pablo Navarro", "Renata Araya", "Sebastian Lagos",
    "Trinidad Molina", "Valentina Saez", "Agustin Bravo", "Catalina Pena", "Domingo Salas", "Emilia Leon",
]

nombres_profesores = [
    "Carolina Gonzalez", "Rafael Mendez", "Patricia Valdes", "Andres Figueroa", "Marcela Cortes", "Ignacio Rivas",
]

bloques_horarios = [
    ("lunes", 8, 10), ("lunes", 10, 12), ("martes", 8, 10), ("martes", 12, 14),
    ("miercoles", 10, 12), ("jueves", 8, 10), ("jueves", 14, 16), ("viernes", 10, 12),
]


def guardar_json(data, ruta):
    Path(ruta).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def generar_datos_semestre(
    semestre: str = "2026-2",
    seed = 20260812,
    n_estudiantes = 24,
    n_solicitudes = 32,
):
    rng = random.Random(seed)
    cursos = []
    for curso in cursos_base:
        cursos.append(dict(curso))

    profesores = []
    for i, nombre in enumerate(nombres_profesores, start=1):
        departamento = "Ciencia de la Computacion" if i % 2 else "Ingeniería Industrial"
        profesores.append({
            "id_persona": f"P{i:03d}",
            "nombre": nombre,
            "email": f"prof{i:03d}@uc.cl",
            "departamento": departamento,
            "max_creditos_docencia": 24,
        })

    cursos_basicos = ["MAT1610", "ICE1514", "IIC1103"]
    carreras = ["Ingenieria", "Ciencia de Datos", "Computacion"]
    estudiantes = []
    for i in range(1, n_estudiantes + 1):
        cantidad_aprobados = rng.randint(0, len(cursos_basicos))
        aprobados = rng.sample(cursos_basicos, cantidad_aprobados)
        estudiantes.append({
            "id_persona": f"E{i:03d}",
            "nombre": nombres_estudiantes[(i - 1) % len(nombres_estudiantes)],
            "email": f"est{i:03d}@uc.cl",
            "carrera": rng.choice(carreras),
            "anio_ingreso": rng.choice([2022, 2023, 2024, 2025]),
            "max_creditos": rng.choice([24, 30, 36]),
            "cursos_aprobados": sorted(aprobados),
        })

    secciones = []
    for curso in cursos:
        cantidad_secciones = rng.choice([1, 2])
        bloques = rng.sample(bloques_horarios, cantidad_secciones)
        for numero, bloque in enumerate(bloques, start=1):
            dia, inicio, fin = bloque
            secciones.append({
                "id_seccion": f"{curso['codigo']}-{numero}",
                "codigo_curso": curso["codigo"],
                "capacidad": rng.randint(5, 9),
                "horarios": [{"dia": dia, "hora_inicio": inicio, "hora_fin": fin}],
                "profesor_id": None,
            })

    datos_iniciales = {
        "semestre": semestre,
        "estudiantes": estudiantes,
        "profesores": profesores,
        "cursos": cursos,
        "secciones": secciones,
    }

    sistema_temporal = sistema_desde_datos(datos_iniciales)
    eventos = []

    for seccion_id in sorted(sistema_temporal.secciones):
        candidatos = list(sistema_temporal.profesores)
        rng.shuffle(candidatos)
        for profesor_id in candidatos:
            ok, _ = sistema_temporal.asignar_profesor(profesor_id, seccion_id)
            if ok:
                eventos.append({"tipo": "asignar_profesor", "profesor_id": profesor_id, "seccion_id": seccion_id})
                break

    contador_solicitudes = 1
    intentos = 0
    ids_estudiantes = list(sistema_temporal.estudiantes)
    ids_secciones = list(sistema_temporal.secciones)
    while len([e for e in eventos if e["tipo"] == "solicitud"]) < n_solicitudes and intentos < n_solicitudes * 80:
        intentos += 1
        activas = []
        for inscripcion in sistema_temporal.inscripciones.values():
            if inscripcion.estado == "activa":
                activas.append(inscripcion)

        if activas and rng.random() < 0.22:
            inscripcion = rng.choice(activas)
            tipo = "cancelar"
            estudiante_id = inscripcion.estudiante_id
            seccion_id = inscripcion.seccion_id
        else:
            tipo = "inscribir"
            estudiante_id = rng.choice(ids_estudiantes)
            seccion_id = rng.choice(ids_secciones)

        solicitud = SolicitudInscripcion(f"S{contador_solicitudes:04d}", estudiante_id, seccion_id, tipo)
        contador_solicitudes += 1
        ok, _ = sistema_temporal.procesar_solicitud(solicitud)
        if ok:
            eventos.append({"tipo": "solicitud", "solicitud": solicitud.to_dict()})

    return {
        "semestre": semestre,
        "seed": seed,
        "parametros": {
            "n_estudiantes": n_estudiantes,
            "n_solicitudes": n_solicitudes,
        },
        "datos_iniciales": datos_iniciales,
        "eventos": eventos,
    }

## 4. Simulador reproducible

parametro max_etapas permite definir la duracion antes de iniciar la ejecucion.

In [19]:
class SimuladorAcademico:
    def __init__(self, configuracion: dict, max_etapas=None):
        self.configuracion = configuracion
        self.max_etapas = max_etapas
        self.sistema = sistema_desde_datos(configuracion["datos_iniciales"])
        eventos = configuracion.get("eventos", [])
        if max_etapas is None:
            self.eventos = eventos
        else:
            self.eventos = eventos[:max_etapas]

    def ejecutar(self, verbose):
        if verbose:
            print(f"Simulacion semestre {self.sistema.semestre}")
            print(f"Etapas a ejecutar: {len(self.eventos)}")

        for numero, evento in enumerate(self.eventos, start=1):
            if evento["tipo"] == "asignar_profesor":
                ok, detalle = self.sistema.asignar_profesor(evento["profesor_id"], evento["seccion_id"])
            elif evento["tipo"] == "solicitud":
                solicitud = solicitud_desde_dict(evento["solicitud"])
                ok, detalle = self.sistema.procesar_solicitud(solicitud)
            else:
                ok = False
                detalle = f"Evento desconocido: {evento['tipo']}"

            if verbose:
                estado = "OK" if ok else "RECHAZADO"
                print(f"[{numero:02d}] {estado}: {detalle}")

        if verbose:
            print("Resumen final:", self.sistema.resumen())
        return self.sistema

    def guardar_reproduccion(self, ruta):
        resultado = {
            "semestre": self.sistema.semestre,
            "seed": self.configuracion.get("seed"),
            "parametros": self.configuracion.get("parametros", {}),
            "max_etapas": self.max_etapas,
            "datos_iniciales": self.configuracion["datos_iniciales"],
            "eventos_planeados": self.configuracion.get("eventos", []),
            "eventos_ejecutados": self.eventos,
            "bitacora": self.sistema.bitacora,
            "estado_final": self.sistema.to_dict(),
        }
        guardar_json(resultado, ruta)
        return resultado


def simulador_desde_json(ruta, max_etapas = None):
    data = json.loads(Path(ruta).read_text(encoding="utf-8"))
    return SimuladorAcademico(data, max_etapas=max_etapas)


def simulador_desde_reproduccion(ruta):
    data = json.loads(Path(ruta).read_text(encoding="utf-8"))
    configuracion = {
        "semestre": data["semestre"],
        "seed": data["seed"],
        "parametros": data["parametros"],
        "datos_iniciales": data["datos_iniciales"],
        "eventos": data["eventos_ejecutados"],
    }
    return SimuladorAcademico(configuracion)

## 5. Consultas

Las consultas trabajan sobre el estado final y la bitacora producida por la simulacion.

In [20]:
class ConsultasAcademicas:
    def __init__(self, sistema):
        self.sistema = sistema

    def cursos_mas_inscritos(self, n=5):
        conteos = {}
        for codigo in self.sistema.cursos:
            conteos[codigo] = 0
        for inscripcion in self.sistema.inscripciones.values():
            if inscripcion.estado == "activa":
                seccion = self.sistema.secciones[inscripcion.seccion_id]
                conteos[seccion.codigo_curso] += 1

        filas = []
        for codigo, total in conteos.items():
            filas.append({"curso": codigo, "nombre": self.sistema.cursos[codigo].nombre, "inscritos": total})
        return sorted(filas, key=lambda x: (-x["inscritos"], x["curso"]))[:n]

    def secciones_menor_disponibilidad(self, n=5):
        filas = []
        for seccion in self.sistema.secciones.values():
            filas.append({
                "seccion": seccion.id_seccion,
                "curso": seccion.codigo_curso,
                "cupos_libres": seccion.cupos_disponibles(),
                "capacidad": seccion.capacidad,
            })
        return sorted(filas, key=lambda x: (x["cupos_libres"], x["seccion"]))[:n]

    def profesores_mayor_carga(self, n=5):
        filas = []
        for profesor in self.sistema.profesores.values():
            cantidad_secciones = 0
            for seccion in self.sistema.secciones.values():
                if seccion.profesor_id == profesor.id_persona:
                    cantidad_secciones += 1
            filas.append({
                "profesor": profesor.nombre,
                "id": profesor.id_persona,
                "creditos": self.sistema.carga_profesor(profesor.id_persona),
                "secciones": cantidad_secciones,
            })
        return sorted(filas, key=lambda x: (-x["creditos"], x["profesor"]))[:n]

    def estudiantes_mas_inscripciones(self, n=5):
        filas = []
        for estudiante in self.sistema.estudiantes.values():
            filas.append({
                "estudiante": estudiante.nombre,
                "id": estudiante.id_persona,
                "activas": len(estudiante.inscripciones_activas),
                "creditos": self.sistema.carga_creditos_estudiante(estudiante.id_persona),
            })
        return sorted(filas, key=lambda x: (-x["activas"], -x["creditos"], x["estudiante"]))[:n]

    def cursos_mas_movimientos(self, n=5):
        conteos = {}
        for codigo in self.sistema.cursos:
            conteos[codigo] = 0
        for registro in self.sistema.bitacora:
            codigo = registro.get("curso")
            if codigo in conteos:
                conteos[codigo] += 1

        filas = []
        for codigo, total in conteos.items():
            filas.append({"curso": codigo, "nombre": self.sistema.cursos[codigo].nombre, "movimientos": total})
        return sorted(filas, key=lambda x: (-x["movimientos"], x["curso"]))[:n]


def imprimir_tabla(titulo: str, filas):
    print(f"\n{titulo}")
    if not filas:
        print("Sin resultados")
        return
    columnas = list(filas[0].keys())
    anchos = {}
    for columna in columnas:
        ancho = len(str(columna))
        for fila in filas:
            ancho = max(ancho, len(str(fila[columna])))
        anchos[columna] = ancho

    print(" | ".join(columna.ljust(anchos[columna]) for columna in columnas))
    print("-+-".join("-" * anchos[columna] for columna in columnas))
    for fila in filas:
        print(" | ".join(str(fila[columna]).ljust(anchos[columna]) for columna in columnas))

## 6. Ejemplo de uso y resultados

genera un archivo de entrada, se ejecuta la simulacion completa y se guarda un archivo final capaz de reproducir exactamente la misma ejecucion.

In [21]:
ruta_datos = Path("oferta_semestre_demo.json")
ruta_resultado = Path("simulacion_resultado_demo.json")

configuracion = generar_datos_semestre(
    semestre="2026-2",
    seed=20260812,
    n_estudiantes=24,
    n_solicitudes=32,
)
guardar_json(configuracion, ruta_datos)

print(f"Archivo de entrada generado: {ruta_datos}")
print(f"Eventos planificados: {len(configuracion['eventos'])}")
print(f"Primeras 3 etapas: {configuracion['eventos'][:3]}")

Archivo de entrada generado: oferta_semestre_demo.json
Eventos planificados: 41
Primeras 3 etapas: [{'tipo': 'asignar_profesor', 'profesor_id': 'P006', 'seccion_id': 'EYP1103-1'}, {'tipo': 'asignar_profesor', 'profesor_id': 'P006', 'seccion_id': 'EYP1103-2'}, {'tipo': 'asignar_profesor', 'profesor_id': 'P004', 'seccion_id': 'ICE1514-1'}]


In [22]:
simulador = simulador_desde_json(ruta_datos)
sistema_final = simulador.ejecutar(verbose=True)
resultado = simulador.guardar_reproduccion(ruta_resultado)

print(f"Archivo de reproduccion generado: {ruta_resultado}")

Simulacion semestre 2026-2
Etapas a ejecutar: 41
[01] OK: Profesor P006 asignado a EYP1103-1
[02] OK: Profesor P006 asignado a EYP1103-2
[03] OK: Profesor P004 asignado a ICE1514-1
[04] OK: Profesor P001 asignado a ICE1514-2
[05] OK: Profesor P001 asignado a IIC1103-1
[06] OK: Profesor P006 asignado a IIC1103-2
[07] OK: Profesor P002 asignado a IIC2133-1
[08] OK: Profesor P002 asignado a IIC2233-1
[09] OK: Profesor P004 asignado a MAT1610-1
[10] OK: E004 inscrito en MAT1610-1
[11] OK: E003 inscrito en IIC1103-2
[12] OK: E009 inscrito en EYP1103-1
[13] OK: E021 inscrito en IIC1103-2
[14] OK: E020 inscrito en ICE1514-1
[15] OK: E004 cancela MAT1610-1
[16] OK: E021 cancela IIC1103-2
[17] OK: E020 cancela ICE1514-1
[18] OK: E012 inscrito en IIC1103-1
[19] OK: E003 inscrito en EYP1103-1
[20] OK: E018 inscrito en EYP1103-2
[21] OK: E016 inscrito en IIC2133-1
[22] OK: E002 inscrito en IIC2233-1
[23] OK: E002 inscrito en MAT1610-1
[24] OK: E009 cancela EYP1103-1
[25] OK: E017 inscrito en ICE15

## 7. Consultas solicitadas

Estas salidas responden las preguntas pedidas usando los datos obtenidos desde la simulacion.

In [23]:
consultas = ConsultasAcademicas(sistema_final)

imprimir_tabla(f"cursos con mayor cantidad de estudiantes inscritos", consultas.cursos_mas_inscritos())
imprimir_tabla(f"secciones con menor disponibilidad de cupos", consultas.secciones_menor_disponibilidad())
imprimir_tabla(f"profesores con mayor carga docente", consultas.profesores_mayor_carga())
imprimir_tabla(f"estudiantes con mayor cantidad de inscripciones", consultas.estudiantes_mas_inscripciones())
imprimir_tabla(f"cursos con mayor numero de movimientos", consultas.cursos_mas_movimientos())


cursos con mayor cantidad de estudiantes inscritos
curso   | nombre                         | inscritos
--------+--------------------------------+----------
EYP1103 | Probabilidad y Estadistica     | 5        
ICE1514 | Dinámica                       | 3        
IIC1103 | Introduccion a la Programacion | 2        
IIC2133 | Estructuras de Datos           | 2        
IIC2233 | Programacion Avanzada          | 2        

secciones con menor disponibilidad de cupos
seccion   | curso   | cupos_libres | capacidad
----------+---------+--------------+----------
EYP1103-1 | EYP1103 | 3            | 6        
IIC2233-1 | IIC2233 | 3            | 5        
IIC2133-1 | IIC2133 | 4            | 6        
EYP1103-2 | EYP1103 | 7            | 9        
ICE1514-2 | ICE1514 | 7            | 9        

profesores con mayor carga docente
profesor          | id   | creditos | secciones
------------------+------+----------+----------
Ignacio Rivas     | P006 | 16       | 3        
Andres Figueroa   | P00

## 8. Reproduccion exacta

La bitacora y el estado final deben coincidir.

In [24]:
replay = simulador_desde_reproduccion(ruta_resultado)
replay.ejecutar(verbose=False)

print(f"Bitacora reproducida exactamente: {replay.sistema.bitacora == resultado['bitacora']}")
print(f"Estado final reproducido exactamente: {replay.sistema.to_dict() == resultado['estado_final']}")

Bitacora reproducida exactamente: True
Estado final reproducido exactamente: True
